In [ ]:
from desc.coils import (CoilSet, FourierPlanarCoil)
from desc.equilibrium import Equilibrium
from desc.plotting import plot_2d, plot_3d, plot_coils, plot_comparison
from desc.grid import LinearGrid
from desc.integrals import compute_B_plasma
import numpy as np
from desc.dipole import _Dipole, DipoleSet, import_dipoles
from desc.io import load
import matplotlib.pyplot as plt
import jax.numpy as jnp
from desc.grid import LinearGrid
from scipy.optimize import minimize, least_squares

In [ ]:
muse_eq = Equilibrium.from_input_file(
    "input_output_files/input.muse-fixedb"
)
muse_dipoles = import_dipoles(
    NFP=2,
    sym=True,
    filename="input_output_files/muse-magnets.focus",
)

In [ ]:
eq=muse_eq

num_coils = 6  # coils per half field period
zeta = np.linspace(0, np.pi / eq.NFP, num_coils, endpoint=False) + np.pi / (2 * eq.NFP * num_coils)
grid = LinearGrid(rho=[0.0], M=0, zeta=zeta, NFP=eq.NFP)
data = eq.axis.compute(["x", "x_s"], grid=grid, basis="rpz")
centers = data["x"]  # center coils on axis position
normals = data["x_s"]  # make normal to coil align with tangent along axis
unique_coils = []
for k in range(num_coils):
    coil = FourierPlanarCoil(
        current=10000,
        center=centers[k, :],
        normal=normals[k, :],
        r_n=0.105,
        basis="rpz",
    )
    unique_coils.append(coil)
coilset = CoilSet(unique_coils, NFP=eq.NFP, sym=eq.sym)

coil_grid = LinearGrid(N=40)

fig = plot_3d(
    eq.surface, "B*n", field=coilset, field_grid=coil_grid, grid=plot_grid
)

fig = plot_coils(coilset, fig=fig)
fig.show()

In [ ]:
axis_data = eq.compute(
    ["|B|", "X", "Y", "Z"],
    grid=grid,
)

axis_pts = np.column_stack([
    np.asarray(axis_data["X"]).ravel(),
    np.asarray(axis_data["Y"]).ravel(),
    np.asarray(axis_data["Z"]).ravel(),
])

B_eq = np.asarray(axis_data["|B|"]).ravel()

I0 = np.array([float(c.current) for c in unique_coils])

def residual(I):
    for c, current in zip(unique_coils, I):
        c.current = current

    B_vec = coilset.compute_magnetic_field(
        axis_pts,
        basis="xyz",
    )

    B_coils = np.linalg.norm(np.asarray(B_vec), axis=-1)

    return B_coils - B_eq

rms_initial = np.sqrt(np.mean(residual(I0)**2))

result = least_squares(
    residual,
    I0,
    method="trf",
    x_scale="jac",
    max_nfev=2000,
    verbose=1,
)

I_opt = result.x

for c, current in zip(unique_coils, I_opt):
    c.current = current

r_final = residual(I_opt)
rms_final = np.sqrt(np.mean(r_final**2))

print("\nOptimization success:", result.success)
print("Message:", result.message)
print(f"Initial RMS error: {rms_initial:.6e} T")
print(f"Final RMS error:   {rms_final:.6e} T")
print("Optimized currents [A]:", I_opt)

B_coils_opt = np.linalg.norm(
    np.asarray(
        coilset.compute_magnetic_field(axis_pts, basis="xyz")
    ),
    axis=-1,
)

In [ ]:
plot_grid = LinearGrid(M=20, N=40, NFP=1, endpoint=True)
axis_grid = LinearGrid(rho=0, theta = 0, N=80, NFP=1, endpoint=True)
fig, ax = plt.subplots(1,1, figsize = (10,6))
data_axis_eq = eq.compute(['|B|','X','Y','Z'], grid = axis_grid)

axis_pts = np.array([data_axis_eq["X"],data_axis_eq["Y"],data_axis_eq["Z"]]).T

ax.plot(data_axis_eq['|B|'], label='|B| eq')
ax.set_title('|B| on axis: Coils vs Eq')

modB_on_axis_coils = np.linalg.norm(coilset.compute_magnetic_field(axis_pts, basis='xyz'), axis = -1)
ax.plot(modB_on_axis_coils,label='|B| Coils')

ax.legend()

In [ ]:
coilset.save('qipm_coils.h5', file_format=None, file_mode='w')